In [1]:
# model=nomic-ai/nomic-embed-text-v1
# text-embeddings-router --model-id $model --port 8080

# curl 127.0.0.1:8080/embed \
#     -X POST \
#     -d '{"inputs":"What is Deep Learning?"}' \
#     -H 'Content-Type: application/json'

In [ ]:
import requests
import pandas as pd 
from tqdm.auto import tqdm

# http://localhost:8080 ...
def emb_local(text):
    r = requests.post(
        "http://localhost:8080/embed", 
        data='{"inputs":"%s"}' % text,
        headers={'Content-Type': 'application/json'}
    )
    try:
        r = r.json()
    except:
        r = None
    return r

In [28]:
full_meeting_data = pd.read_json('cache/2024-03-25__retrieve-examples-for-demo/full-meeting-data-with-headers.jsonl', lines=True)

In [71]:
all_speaker_text = full_meeting_data['transcribed_text'].explode().str.get('text').str.strip().drop_duplicates()

tqdm.pandas()
public_comment_speaker_text =  (
    full_meeting_data
        .loc[lambda df: df['header'] == 'PUBLIC COMMENT']
        ['transcribed_text']
        .explode()
        .str.get('text')
        .str.strip()
        .drop_duplicates()
)

In [76]:
all_embeddings = public_comment_speaker_text.progress_apply(emb_local)

  0%|          | 0/101753 [00:00<?, ?it/s]

In [104]:
all_embeddings_proc = all_embeddings.str.get(0)
all_embeddings_proc = public_comment_speaker_text.to_frame().assign(embs=all_embeddings_proc)
all_embeddings_proc = (full_meeting_data
        .loc[lambda df: df['header'] == 'PUBLIC COMMENT']
        ['transcribed_text']
        .explode()
        .str.get('text')
        .to_frame()
        .pipe(lambda df: df.merge(all_embeddings_proc, on='transcribed_text', how='left').set_index(df.index))
)

In [113]:
(all_embeddings_proc
     # .to_json('cache/2024-03-25__retrieve-examples-for-demo/public-speaker-embeddings.jsonl', lines=True, orient='records')
)

,transcribed_text,embs
5,Any public speaker using translation assistanc...,"[0.016878756, 0.092349365, -0.010085594, -0.03..."
5,The President or the Board may limit the total...,"[0.032559536, 0.08584117, -0.0075005568, 0.001..."
5,"Additionally, if members of the public in the...",NaN
5,"And finally, if by majority vote the Board cho...","[0.011130428, 0.03996308, -0.011185022, -0.000..."
5,"Thank you, Madam Clerk.",NaN
...,...,...
32786,Public comment is closed.,"[0.010215016, -0.023977626, -0.012034441, -0.0..."
32786,"Otto, would you like to make in-person public ...","[-0.008282392, -0.00069672015, -0.020344831, -..."
32786,No.,"[-0.016971264, 0.016726913, -0.00037763285, -0..."
32786,Public comment is closed.,"[0.010215016, -0.023977626, -0.012034441, -0.0..."


# Try to do better news article matching 

In [114]:
final_matching_df = pd.read_csv('cache/final-matching-articles-and-meetings.csv', index_col=0)

In [119]:
ls ../data/articles/#sfchron-fetched-articles.jsonl

josh-shapiro-pa-governor-bipartisanship-20221226.html
latimes-8-years.jsonl
latimes-article-urls-to-fetch.csv
latimes-cc-articles-to-fetch-unique.jsonl.gz
latimes-cc-articles-to-fetch.jsonl.gz
latimes-output-round-2.jsonl
sfchron-article-urls-to-fetch.csv.gzip
sfchron-cc-fetched-articles-2.jsonl.gz
sfchron-cc-fetched-articles.jsonl
sfchron-cc-to-fetch.jsonl.gz
sfchron-fetched-articles-with-html.jsonl.gz
sfchron-fetched-articles.jsonl
sfchron-fetched-urls.jsonl


In [121]:
pd.read_json('../data/articles/sfchron-fetched-articles.jsonl', lines=True, nrows=5)

,article_url,homepage_key,article_text,article_publish_date,article_authors,article_top_image,article_video,article_wayback_timestamp,all_article_wayback_timestamps
0,http://www.sfchronicle.com/politics/article/Ju...,20130325002427,Kris Perry (l to r) and Sandy Stier pose for a...,None,[Carolyn Lochhead],https://web.archive.org/web/20130327050715im_/...,[],20130327050715,"[20130327050715, 20130421171608, 2013052117515..."
1,http://www.sfchronicle.com/education/article/M...,20130325002427,Students in Patricia Juri's fourth-grade class...,None,[Jill Tucker],https://web.archive.org/web/20130327030624im_/...,[],20130327030624,"[20130327030624, 20130713121224, 2013101903434..."
2,http://www.sfchronicle.com/news/article/Introd...,20130325002427,Activated? Sign In\n\nIf you've already activa...,None,[],https://web.archive.org/web/20130330105451im_/...,[],20130330105451,"[20130330105451, 20130419043650, 2013053013033..."
3,http://www.sfchronicle.com/crime/article/Game-...,20130325002427,Photo illustration for violence in video games...,None,[Benny Evangelista],https://web.archive.org/web/20130327023910im_/...,[],20130327023910,"[20130327023910, 20160604140122]"
4,http://www.sfchronicle.com/bayarea/article/Oak...,20130325002427,Oakland Raider cheerleaders welcomed 5K runner...,None,[Julian Guthrie],https://web.archive.org/web/20130329013952im_/...,[],20130329013952,[20130329013952]
